In [ ]:
# 检查/打印关键依赖库的版本号，确保运行环境满足要求
from importlib.metadata import version

# 定义需要检测版本的依赖包名称列表
pkgs = [
    #"blobfile",         # to download pretrained weights
    # 注意：这里列出 tiktoken 仅用于打印其版本号，但下方实际的分词器实现使用的是 `tokenizers` 库
    # (见后面 `from tokenizers import Tokenizer`)，二者并非同一个包——这是文档/依赖列表与实际代码不一致的风险点，此处仅标注、不修改原代码
    "huggingface_hub",  # to download pretrained weights
    "tiktoken",         # to implement the tokenizer
    "torch",            # to implement the model
]
# 遍历依赖包名，逐个打印当前已安装的版本号，便于复现/排查环境问题
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 配置要下载的 Hugging Face 仓库 ID 和本地缓存目录
from pathlib import Path

# 选择要使用的 tiny-aya 变体仓库（global/fire/water/earth 是不同的模型变体），默认用 global，其余通过注释切换
REPO_ID = "CohereLabs/tiny-aya-global"
#REPO_ID = "CohereLabs/tiny-aya-fire"
#REPO_ID = "CohereLabs/tiny-aya-water"
#REPO_ID = "CohereLabs/tiny-aya-earth"

# 取 REPO_ID 最后一段作为本地目录名，用来缓存下载下来的 tokenizer 与权重文件
LOCAL_DIR = Path(REPO_ID).parts[-1]

In [ ]:
# ===== 模型核心组件定义：FeedForward、CohereLayerNorm、RoPE、GQA（分组查询注意力）、TransformerBlock、TinyAyaModel、KVCache =====
import torch
import torch.nn as nn


# SwiGLU 前馈网络（门控线性单元变体）
# 结构：x -> [fc1(x) 经 SiLU 激活] 逐元素乘以 [fc2(x)] -> fc3 输出；三个线性层均不带 bias
# 维度：输入/输出为 emb_dim，中间隐藏层为 hidden_dim
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    # 前向传播：x 形状为 (batch_size, seq_len, emb_dim)
    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        # SiLU(fc1(x)) 作为门控，逐元素乘以 fc2(x)，即 SwiGLU 激活；形状为 (batch_size, seq_len, hidden_dim)
        x = nn.functional.silu(x_fc1) * x_fc2
        # fc3 把隐藏维度 hidden_dim 投影回 emb_dim，输出形状 (batch_size, seq_len, emb_dim)
        return self.fc3(x)
# Aya uses a bias-less LayerNorm variant.
# The difference to classic LayerNorm is that it only
# has a scale parameter (weight), no shift parameter (bias).

# 补充说明：这里实现的其实是“无偏置(bias-less)的 LayerNorm”，不是 RMSNorm。
# RMSNorm（如 Llama/Gemma 中使用）不减均值、只用均方根做缩放；
# 而这里的 CohereLayerNorm 依然会减去均值、除以标准差(标准 LayerNorm 做法)，只是省略了可学习的偏置(bias)/平移(shift)参数，仅保留缩放参数 weight
class CohereLayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))

    # 前向传播：x 可以是 (batch, seq_len, emb_dim) 或 (batch, num_heads, seq_len, head_dim)，归一化都在最后一维上进行
    def forward(self, x):
        input_dtype = x.dtype
        # 先转成 float32 计算，避免在 bfloat16/float16 下因数值范围不足导致归一化不稳定
        x = x.to(torch.float32)
        # 计算最后一维的均值（标准 LayerNorm 做法；RMSNorm 会跳过这一步）
        mean = x.mean(dim=-1, keepdim=True)
        # 计算方差（基于减去均值后的偏差平方的均值）
        variance = (x - mean).pow(2).mean(dim=-1, keepdim=True)
        # 标准化：减均值、除以标准差（加 eps 防止除零）
        x = (x - mean) * torch.rsqrt(variance + self.eps)
        # 只做缩放(weight)，不做平移(无 bias)；最后转换回输入原始 dtype
        return (self.weight.to(torch.float32) * x).to(input_dtype)
# ----- RoPE（旋转位置编码）参数预计算 -----
# 预先算好每个位置、每个频率对应的 cos/sin 值，供 apply_rope 在前向传播时直接查表使用
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "head_dim must be even"

    # Compute the inverse frequencies
    # 计算逆频率 inv_freq，长度为 head_dim // 2，频率随维度呈指数衰减（theta_base 的幂）
    inv_freq = 1.0 / (
        theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim)
    )
    # 生成位置序列 0 ... context_length-1，用于计算每个绝对位置对应的旋转角度
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    # 位置 (context_length, 1) 与逆频率 (1, head_dim//2) 广播相乘，得到每个位置在每个频率上的旋转角度
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # ----- 交织(interleaved)布局说明 -----
    # Cohere uses interleaved even/odd angle layout per head-dim pair.
    # Llama2 notebook examples often use a split-halves layout via cat([angles, angles]).
    # Both are equivalent only when paired with the matching rotate logic:
    # - interleaved layout -> even/odd rotation implementation (below)
    # - split-halves layout -> half/half rotate implementation
    # 将 (context_length, head_dim//2) 的角度逐元素重复一次，交织成 (context_length, head_dim)，
    # 即变成 [a0, a0, a1, a1, ...]，与下面 apply_rope 中偶数/奇数分量一一对应
    angles = torch.repeat_interleave(angles, 2, dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    # 返回 cos、sin，形状均为 (context_length, head_dim)；前向时按位置切片使用
    return torch.cos(angles), torch.sin(angles)

# ----- 应用 RoPE：对 Q/K 做旋转变换 -----
def apply_rope(x, cos, sin, offset=0):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "head_dim must be even"

    # Split x into even and odd components (interleaved layout)
    # 交织布局下，偶数下标与奇数下标分别对应旋转公式里的一对 (x_i, x_{i+1})
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]

    # Adjust sin and cos shapes
    # 根据起始位置 offset（KV cache 场景下等于当前已生成 token 的绝对位置）截取对应长度的 cos/sin，
    # 并 unsqueeze 扩展维度，便于和 x 的 (batch, num_heads, seq_len, head_dim) 广播
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    x_float = x.float()
    # 构造旋转分量：对每一对 (x_even, x_odd) 重新交织成 (-x_odd, x_even)，对应旋转矩阵的另一分量
    rotated = torch.stack((-x_odd.float(), x_even.float()), dim=-1).flatten(-2)
    # RoPE 核心公式：x_rotated = x * cos + rotate(x) * sin，逐位置、逐维度施加旋转
    x_rotated = (x_float * cos) + (rotated * sin)

    # 转回原始 dtype（如 bfloat16），避免精度提升带来额外显存/速度开销
    return x_rotated.to(dtype=x.dtype)
# ----- 分组查询注意力 (Grouped-Query Attention, GQA) -----
# GQA 是多头注意力(MHA)与多查询注意力(MQA)之间的折中：
# Query 仍使用 num_heads 个头，但 Key/Value 只用较少的 num_kv_groups 个头（组），
# 组内的多个 Query 头共享同一组 K/V，从而显著减少 KV cache 的显存占用
class GroupedQueryAttention(nn.Module):
    def __init__(
        self,
        d_in,
        num_heads,
        num_kv_groups,
        head_dim=None,
        qk_norm=False,
        attention_bias=False,
        dtype=None,
        attn_type="full_attention",
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        # group_size 表示每个 KV 组要被多少个 Query 头共享
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        # d_out 是多头拼接后的总维度 = num_heads * head_dim，可能与输入 d_in 不同
        self.d_out = num_heads * head_dim
        self.attn_type = attn_type

        # Q 投影：输出维度为 num_heads * head_dim（每个头独立的 Q）
        self.W_query = nn.Linear(
            d_in,
            self.d_out,
            bias=attention_bias,
            dtype=dtype,
        )
        # K 投影：输出维度为 num_kv_groups * head_dim（远小于 Q，因为 K 的头数更少）
        self.W_key = nn.Linear(
            d_in,
            num_kv_groups * head_dim,
            bias=attention_bias,
            dtype=dtype,
        )
        # V 投影：与 K 相同，维度为 num_kv_groups * head_dim
        self.W_value = nn.Linear(
            d_in,
            num_kv_groups * head_dim,
            bias=attention_bias,
            dtype=dtype,
        )
        # 输出投影：把拼接后的多头结果 (num_heads*head_dim) 映射回 d_in
        self.out_proj = nn.Linear(
            self.d_out,
            d_in,
            bias=attention_bias,
            dtype=dtype,
        )

        # 可选的 QK-Norm：对每个头的 Q/K 在 head_dim 维度上做归一化，用于稳定训练/推理数值
        # 本 notebook 中 TransformerBlock 固定传入 qk_norm=False，因此该分支实际未启用
        if qk_norm:
            self.q_norm = CohereLayerNorm(head_dim, eps=1e-6)
            self.k_norm = CohereLayerNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    # 前向传播输入说明：
    # x: (batch_size, num_tokens, d_in)
    # mask: 对应 attn_type 的因果/滑窗掩码，可广播到 (batch, heads, num_tokens, total_kv_tokens)
    # cos, sin: RoPE 预计算表
    # start_pos: 这批 token 在整个序列中的起始绝对位置（KV cache 场景下用于对齐 RoPE 相位）
    # cache: 可选的 (prev_keys, prev_values)，来自 KVCache，用于增量解码
    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        # 拆成多头：(b, num_tokens, num_heads, head_dim) 再转置为 (b, num_heads, num_tokens, head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        # 注意 K/V 只有 num_kv_groups 个头，而不是 num_heads 个头，这正是 GQA 节省显存的关键
        keys_new = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys_new = self.k_norm(keys_new)

        # Cohere2 架构特点：只有滑动窗口注意力层(sliding_attention)使用 RoPE 位置编码，
        # 全局注意力层(full_attention)不加位置编码(NoPE)，依赖滑窗层已编码的位置信息与因果结构
        # Cohere2 applies RoPE only on sliding-attention layers.
        if self.attn_type == "sliding_attention":
            # 对本次新计算出的 Q/K 施加旋转位置编码，offset=start_pos 保证 KV cache 场景下角度与绝对位置对齐
            queries = apply_rope(queries, cos, sin, offset=start_pos)
            keys_new = apply_rope(keys_new, cos, sin, offset=start_pos)

        # ----- KV Cache 拼接 -----
        # 注意：这里拼接的是（若为 sliding_attention 层）已经过 RoPE 旋转后的 keys_new，
        # 即缓存中保存的是“旋转后”的 K，而不是旋转前的原始投影结果
        if cache is not None:
            # prev_k/prev_v 形状均为 (b, num_kv_groups, 已缓存的历史长度, head_dim)
            prev_k, prev_v = cache
            # 在序列长度维度(dim=2)上拼接历史 K 与新 K，得到完整的 K 缓存（V 同理）
            keys = torch.cat([prev_k, keys_new], dim=2)
            values = torch.cat([prev_v, values_new], dim=2)
            next_cache = (keys, values)
        else:
            keys, values = keys_new, values_new
            next_cache = (keys, values)

        # 把 K/V 从 num_kv_groups 个头，通过 repeat_interleave 复制扩展到 num_heads 个头，
        # 使每个 Query 头都能找到与之对应（组内共享）的 K/V，形状变为 (b, num_heads, total_kv_len, head_dim)
        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # Attention
        # 计算注意力得分：Q @ K^T，形状 (b, num_heads, num_tokens, total_kv_len)
        attn_scores = queries @ keys.transpose(2, 3)
        # 用布尔掩码 mask 把不允许看到的位置（未来位置 / 超出滑窗范围 / padding）填为 -inf
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)

        # 按 sqrt(head_dim) 缩放后做 softmax；用 float32 计算 softmax 保证数值稳定，再转回原 dtype
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1, dtype=torch.float32).to(queries.dtype)
        # 加权求和得到上下文向量，再把多头结果拼接回 (b, num_tokens, d_out)
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)

        # 输出投影回 d_in 维度；同时把本层最新的 KV cache 一并返回，供上层保存
        return self.out_proj(context), next_cache
# ----- Transformer 块：并行残差结构 (Parallel Residual Block) -----
# 不同于常见的串行 Pre-Norm 结构 (x + Attn(Norm(x))，再 + FFN(Norm(x)))，
# Cohere2 这里对同一个归一化后的输入 x 同时计算注意力分支和前馈分支，两者相加后再加回残差
class TransformerBlock(nn.Module):
    def __init__(self, cfg, attn_type):
        super().__init__()
        self.attn_type = attn_type

        # 每一层根据配置的 attn_type ('sliding_attention' 或 'full_attention') 决定使用滑窗掩码还是全局因果掩码
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_heads"],
            head_dim=cfg["head_dim"],
            qk_norm=False,
            attention_bias=cfg["attention_bias"],
            dtype=cfg["dtype"],
            attn_type=attn_type,
        )
        self.ff = FeedForward(cfg)
        # 注意：只有一个 input_layernorm，注意力分支和前馈分支共享同一份归一化输出，
        # 不像标准 Pre-Norm Transformer 那样在 FFN 前再做一次归一化
        self.input_layernorm = CohereLayerNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])

    # 前向传播需同时传入全局掩码和局部(滑窗)掩码，内部根据 self.attn_type 选择使用哪一个
    def forward(self, x, mask_global, mask_local, cos, sin, start_pos=0, cache=None):
        attn_mask = mask_local if self.attn_type == "sliding_attention" else mask_global

        shortcut = x
        # 对输入做一次归一化，注意力和前馈分支都基于这个同一份归一化结果计算
        x = self.input_layernorm(x)
        x_attn, next_cache = self.att(
            x,
            attn_mask,
            cos,
            sin,
            start_pos=start_pos,
            cache=cache,
        )  # Shape [batch_size, num_tokens, emb_dim]
        # 注意这里前馈分支的输入是归一化后的 x（同一个 x），而不是注意力的输出，体现“并行”结构
        x_ff = self.ff(x)

        # Cohere2 parallel residual block
        # 残差连接：原始输入 + 注意力输出 + 前馈输出，三者相加（并行残差）
        x = shortcut + x_attn + x_ff
        return x, next_cache
# ----- 完整模型：TinyAyaModel -----
# 负责组装词嵌入、多层 TransformerBlock（滑窗/全局交替）、最终归一化、输出头，
# 并管理 RoPE 参数缓冲区、KV cache 相关的位置计数器 (current_pos)
class TinyAyaModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert len(cfg["layer_types"]) == cfg["n_layers"], "layer_types must match n_layers"

        self.cfg = cfg

        # 词嵌入表：(vocab_size, emb_dim)
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        # 按 cfg['layer_types'] 中每一层指定的类型（滑窗/全局）构建对应的 TransformerBlock
        self.trf_blocks = nn.ModuleList([TransformerBlock(cfg, t) for t in cfg["layer_types"]])

        self.final_norm = CohereLayerNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        self.logit_scale = cfg["logit_scale"]

        # 预先计算好覆盖最大 context_length 的 RoPE cos/sin 查找表，注册为非持久化 buffer（不参与 state_dict 保存）
        cos, sin = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        # 权重绑定 (weight tying)：输出头权重直接复用词嵌入权重，减少参数量，是此类模型的常见做法
        if cfg["tie_word_embeddings"]:
            self.out_head.weight = self.tok_emb.weight

        # current_pos 记录增量解码时已写入 KV cache 的绝对位置，用于下一次调用时确定新 token 的起始位置
        self.current_pos = 0  # Track current position in KV cache

    # 构造两种因果掩码：
    # 1) mask_global：标准因果掩码，只允许看到 <= 当前查询位置的 key
    # 2) mask_local：在 mask_global 基础上，再屏蔽掉超出 sliding_window 范围的“太久远”的 key
    # 掩码中 True 表示该位置需要被屏蔽
    def create_masks(self, num_tokens, device, pos_start=0, total_kv_tokens=None):
        if total_kv_tokens is None:
            total_kv_tokens = pos_start + num_tokens

        # 当前这批查询 token 的绝对位置，形状 (num_tokens, 1)
        query_positions = torch.arange(pos_start, pos_start + num_tokens, device=device).unsqueeze(1)
        # 所有需要关注的 key 的绝对位置（含历史缓存），形状 (1, total_kv_tokens)
        key_positions = torch.arange(total_kv_tokens, device=device).unsqueeze(0)

        # Future mask
        # key 位置大于 query 位置即为“未来”，需要屏蔽——标准自回归因果掩码
        mask_global = key_positions > query_positions

        # Sliding-window mask
        # 如果 key 位置比 query 位置早了超过 sliding_window 步，也视为“太久远”，需要屏蔽
        far_past = key_positions + self.cfg["sliding_window"] <= query_positions
        # 滑窗掩码 = 因果掩码 或 太久远，两者任一为 True 就屏蔽
        mask_local = mask_global | far_past

        # Expand to [batch, heads, seq, seq]-broadcastable shape
        # 增加 batch 和 head 两个维度 (unsqueeze(0).unsqueeze(0))，便于与 (batch, heads, seq, seq) 的注意力分数广播
        return mask_global.unsqueeze(0).unsqueeze(0), mask_local.unsqueeze(0).unsqueeze(0)

    # 模型整体前向传播：
    # input_ids: (batch_size, num_tokens)——使用 KV cache 时，通常每步只传入 1 个新 token
    # attention_mask: 可选的 padding 掩码 (1=有效 token, 0=padding)
    # cache: 可选的 KVCache 实例，非 None 时开启增量解码模式
    def forward(self, input_ids, attention_mask=None, cache=None):
        tok_embeds = self.tok_emb(input_ids)
        x = tok_embeds
        num_tokens = x.shape[1]

        # 增量解码模式：本次新 token 的起始位置 = 之前已缓存的绝对位置 current_pos
        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + num_tokens
            self.current_pos = pos_end
            total_kv_tokens = pos_end
        # 非缓存模式（比如首次前向/整段输入）：从位置 0 开始，key 的总长度就是本次输入长度
        else:
            pos_start = 0
            total_kv_tokens = num_tokens

        # 根据当前位置区间和 KV 总长度构造对应的因果/滑窗掩码
        mask_global, mask_local = self.create_masks(
            num_tokens,
            x.device,
            pos_start=pos_start,
            total_kv_tokens=total_kv_tokens,
        )

        # 如果提供了 padding mask，需要把 padding 位置也一并加入两种掩码中屏蔽掉
        if attention_mask is not None:
            # True means mask in this implementation.
            pad_mask = attention_mask[:, None, None, :total_kv_tokens].to(dtype=torch.bool).logical_not()
            mask_global = mask_global | pad_mask
            mask_local = mask_local | pad_mask

        # 把预计算好的 RoPE 表转换到当前输入所在的 device 和 dtype
        cos = self.cos.to(x.device, dtype=x.dtype)
        sin = self.sin.to(x.device, dtype=x.dtype)

        # 依次通过每一层 TransformerBlock，取出/更新该层对应的 KV cache
        for i, block in enumerate(self.trf_blocks):
            blk_cache = cache.get(i) if cache else None
            x, new_blk_cache = block(
                x,
                mask_global,
                mask_local,
                cos,
                sin,
                start_pos=pos_start,
                cache=blk_cache,
            )
            if cache is not None:
                cache.update(i, new_blk_cache)

        # 所有层结束后做最终归一化
        x = self.final_norm(x)
        # 投影到词表维度得到 logits，注意先转换回 cfg['dtype'] 精度
        logits = self.out_head(x.to(self.cfg["dtype"]))
        # Cohere 系列模型会对最终 logits 做一个缩放 (logit_scale)；此处配置为 1.0，相当于不缩放
        return logits * self.logit_scale

    # 重置位置计数器，开始新一轮独立的增量解码（例如新的一轮对话/新的 prompt）
    def reset_kv_cache(self):
        self.current_pos = 0


# ----- 简单的 KV Cache 容器 -----
# 按层 (layer_idx) 存储每一层的 (keys, values) 元组，避免增量解码时重复计算历史 token 的注意力
class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None

2. Initialize model

In [ ]:
# ===== 模型超参数配置（对应 tiny-aya / Cohere2 架构）=====
TINY_AYA_CONFIG = {
    "vocab_size": 262_144,            # Vocabulary size
    "context_length": 500_000,        # Context length in the HF config
    "emb_dim": 2048,                  # Embedding dimension
    "n_heads": 16,                    # Number of attention heads
    "n_layers": 36,                   # Number of layers
    "hidden_dim": 11_008,             # Size of the intermediate dimension in FeedForward
    "head_dim": 128,                  # Size of the heads in GQA
    "n_kv_heads": 4,                  # Number of KV heads for grouped-query attention
    "attention_bias": False,          # Whether attention projections use bias terms
    "attention_dropout": 0.0,         # Attention dropout
    # 注意：context_length 高达 50 万，但大部分层 (sliding_attention) 的注意力窗口被限制在 sliding_window=4096，
    # 只有间隔出现的 full_attention 层才能看到超出滑窗范围的全部历史，这是长上下文 + 低计算成本的关键设计
    "sliding_window": 4096,           # Sliding-window attention context
    # 逐层指定注意力类型：每 4 层里前 3 层为滑窗注意力、第 4 层为全局注意力，如此循环（36 层共 9 组）
    "layer_types": [
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
    ],
    "rope_base": 50_000.0,            # The base in RoPE's "theta"
    "layer_norm_eps": 1e-5,           # Epsilon used by layer normalization
    "logit_scale": 1.0,               # Final logits scaling factor
    "tie_word_embeddings": True,      # Whether input embedding and output head are tied
    "bos_token_id": 2,
    "eos_token_id": 3,
    "pad_token_id": 0,
    "dtype": torch.bfloat16,          # Lower-precision dtype to reduce memory usage
}
# 根据上面的配置实例化模型（此时权重是随机初始化的，还没有加载预训练权重）
model = TinyAyaModel(TINY_AYA_CONFIG)
# 估算模型在给定 dtype 下所需的显存/内存大小（参数 + 梯度占位 + 缓冲区）
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

# 对比不同精度下的模型体积：bfloat16 相比 float32 能减少约一半内存占用
print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
# 统计模型总参数量（注意：由于权重绑定，tok_emb 与 out_head 共享同一份权重张量，parameters() 会把它统计两次）
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
# 由于 tie_word_embeddings=True，tok_emb 和 out_head 共享同一份权重，parameters() 里被重复计入了一次，
# 因此这里减去一次 tok_emb 的参数量，得到“去重后”的真实参数量
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 自动选择可用的计算设备：优先 CUDA GPU，其次 Apple Silicon 的 MPS，否则回退到 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 把模型所有参数/缓冲区搬到选定设备上；末尾分号用于在 Notebook 中抑制该表达式的输出
model.to(device);

3. Load tokenizer

In [ ]:
# 使用 Hugging Face 的 tokenizers 库（Rust 实现的快速分词器），而不是 tiktoken，加载 tokenizer.json 文件
from tokenizers import Tokenizer


# 对 tokenizers.Tokenizer 的简单封装，暴露 encode/decode 接口，并解析特殊 token 的 id
class TinyAyaTokenizer:
    def __init__(self, tokenizer_file_path, eos_token_id=3, pad_token_id=0, bos_token_id=2):
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))

        # 优先从分词器自带词表中查找特殊 token 对应的 id，找不到则退回构造函数传入的默认值
        eos_from_tok = self._tok.token_to_id("<EOS_TOKEN>")
        pad_from_tok = self._tok.token_to_id("<PAD>")
        bos_from_tok = self._tok.token_to_id("<BOS_TOKEN>")

        self.eos_token_id = eos_from_tok if eos_from_tok is not None else eos_token_id
        self.pad_token_id = pad_from_tok if pad_from_tok is not None else pad_token_id
        self.bos_token_id = bos_from_tok if bos_from_tok is not None else bos_token_id

    # 文本 -> token id 列表
    def encode(self, text):
        return self._tok.encode(text).ids

    # token id 列表 -> 文本；skip_special_tokens=False 保留特殊 token，便于调试查看模型是否正确停止
    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)


# 按 Cohere Command/Aya 系列的对话模板格式，把用户输入包装成模型期望的 prompt 文本
# 依次是：起始符 -> 对话轮次开始+用户角色标记 -> 用户文本 -> 轮次结束标记 -> 对话轮次开始+机器人角色标记+响应起始标记
def apply_chat_template(user_text):
    return (
        "<BOS_TOKEN>"
        "<|START_OF_TURN_TOKEN|><|USER_TOKEN|>"
        f"{user_text}"
        "<|END_OF_TURN_TOKEN|>"
        "<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>"
    )

In [ ]:
# 打印当前选择的仓库 ID，确认将要使用/下载的是哪一个 tiny-aya 变体模型（注：本行原文按字符逐个拆分存储，属于原始格式，保持不变）
print(REPO_ID)

In [ ]:
# 登录 Hugging Face Hub（部分仓库为受限访问，需要账号授权后才能下载）
from huggingface_hub import login
login()
from huggingface_hub import hf_hub_download

# 若本地尚未缓存 tokenizer.json，则尝试从 Hub 下载；下载失败则回退到相对路径，交由后续报错处理
tokenizer_file_path = Path(LOCAL_DIR) / "tokenizer.json"
if not tokenizer_file_path.exists():
    try:
        tokenizer_file_path = hf_hub_download(repo_id=REPO_ID, filename="tokenizer.json", local_dir=LOCAL_DIR)
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")
        tokenizer_file_path = "tokenizer.json"
# 用下载好的/本地已有的 tokenizer.json 及配置中的特殊 token id 构造分词器实例
tokenizer = TinyAyaTokenizer(
    tokenizer_file_path=Path(LOCAL_DIR) / "tokenizer.json",
    eos_token_id=TINY_AYA_CONFIG["eos_token_id"],
    pad_token_id=TINY_AYA_CONFIG["pad_token_id"],
    bos_token_id=TINY_AYA_CONFIG["bos_token_id"],
)

# 构造一个对话式 prompt，并做一次 编码->解码 的往返测试，确认分词器工作正常
prompt = apply_chat_template("Give me a short introduction to large language models in 3 sentences.")
input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

4. Load pretrained weights

In [ ]:
# ===== 将 Hugging Face 格式（Cohere2）的预训练权重加载进自定义的 TinyAyaModel =====
def load_weights_into_tiny_aya(model, param_config, params):
    # 辅助函数：校验形状一致后，把 right 的数据拷贝进 left 对应的 dtype/device，并返回该参数（便于链式赋值）
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right.to(dtype=left.dtype, device=left.device))
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # 加载词嵌入权重
    model.tok_emb.weight = assign(
        model.tok_emb.weight,
        params["model.embed_tokens.weight"],
        "model.embed_tokens.weight",
    )

    # 逐层加载每个 TransformerBlock 里的注意力、前馈、归一化权重
    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight",
        )

        # Feedforward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )

        # Layernorm
        block.input_layernorm.weight = assign(
            block.input_layernorm.weight,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight",
        )

    # Final normalization and output head
    # 加载最终归一化层权重
    model.final_norm.weight = assign(
        model.final_norm.weight,
        params["model.norm.weight"],
        "model.norm.weight",
    )

    # 如果权重文件里显式提供了独立的 lm_head 权重，就直接加载；
    # 否则说明该模型采用权重绑定，直接复用词嵌入权重作为输出头权重
    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        if param_config["tie_word_embeddings"]:
            model.out_head.weight = model.tok_emb.weight
            print("Model uses weight tying.")
# ----- 下载模型权重分片并合并，然后调用上面的函数完成加载 -----
import json
from safetensors.torch import load_file
from huggingface_hub import snapshot_download


# 下载整个仓库快照（包含所有 safetensors 分片和 index 文件）到本地目录
repo_dir = snapshot_download(repo_id=REPO_ID, local_dir=LOCAL_DIR)
# 读取 safetensors 分片索引，其中记录了每个参数名对应存放在哪个分片文件里
index_path = Path(repo_dir) / "model.safetensors.index.json"
with open(index_path, "r") as f:
    index = json.load(f)

# 依次加载所有分片文件，并合并成一个完整的 { 参数名: 张量 } 字典
weights_dict = {}
for filename in sorted(set(index["weight_map"].values())):
    shard_path = Path(repo_dir) / filename
    shard = load_file(shard_path)
    weights_dict.update(shard)

# 执行真正的权重加载
load_weights_into_tiny_aya(model, TINY_AYA_CONFIG, weights_dict)
# 加载权重后（张量此前在 CPU 上），把模型重新搬到目标计算设备
model.to(device)
# 释放合并后的权重字典，及时回收内存（权重已拷贝进模型参数里，不再需要这份副本）
del weights_dict

In [ ]:
# 统计模型中“物理上独立”的参数量：通过 data_ptr() 判断底层存储是否相同，
# 从而正确处理权重绑定 (tied weights) 场景，避免重复计数共享的存储
def count_unique_parameters(model):
    unique_params = set()
    total_unique_params = 0

    for param in model.parameters():
        if param.data_ptr() not in unique_params:
            total_unique_params += param.numel()
            unique_params.add(param.data_ptr())

    return total_unique_params

# 应与前面 cell 5 中计算得到的 total_params_normalized 一致，用作交叉验证
total_params_uniq = count_unique_parameters(model)
print(f"Total number of unique parameters: {total_params_uniq:,}")

5. Generate text

In [ ]:
# ===== 基于 KV Cache 的贪心解码（流式生成）=====
# 收集需要在生成时提前停止的特殊 token id：EOS、<|END_RESPONSE|>、<|END_OF_TURN_TOKEN|>
stop_ids = {
    tokenizer.eos_token_id,
    tokenizer._tok.token_to_id("<|END_RESPONSE|>"),
    tokenizer._tok.token_to_id("<|END_OF_TURN_TOKEN|>"),
}
# 过滤掉词表中不存在、返回 None 的 token id
stop_ids = {x for x in stop_ids if x is not None}


# 逐 token 生成的生成器函数：
# - 首次调用把完整 prompt 一次性喂给模型，填充所有层的 KV cache（"prime" 阶段）
# - 之后每一步只需把上一步新采样出的单个 token 喂给模型，配合 cache 增量计算注意力，
#   避免每步都重新计算整个历史序列的注意力，这正是 KV cache 加速自回归生成的核心
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens,
    stop_token_ids=None,
    context_size=None,
):
    stop_token_ids = set(stop_token_ids or [])

    # 切换到 eval 模式并关闭梯度计算，纯推理场景不需要反向传播
    model.eval()
    with torch.no_grad():
        # 为模型的每一层创建一个空的 KV cache 槽位
        cache = KVCache(n_layers=model.cfg["n_layers"])
        # 重置模型内部的位置计数器 current_pos，确保从 0 开始计数
        model.reset_kv_cache()

        # Prime the cache with the initial context
        logits = model(token_ids, cache=cache)

        for _ in range(max_new_tokens):
            # 贪心解码：直接选取概率最大（logits 最大）的 token，不做采样
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)

            # 命中任一停止符则结束生成
            if stop_token_ids and next_token.item() in stop_token_ids:
                break

            # 把新生成的 token 向外“流式”产出，调用方可以边生成边打印
            yield next_token

            # 维护完整的 token_ids 序列（主要用于记录/调试，模型下一步并不需要整段输入）
            token_ids = torch.cat([token_ids, next_token], dim=1)
            # Feed only the new token to the model; cache handles history
            logits = model(next_token, cache=cache)
# ----- 实际运行一次生成，并流式打印结果 -----
prompt = apply_chat_template("Give me a short introduction to large language models in 3 sentences.")
input_token_ids = tokenizer.encode(prompt)
# 编码 prompt 并转成形状为 (1, prompt_len) 的张量 (batch_size=1)
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


# 重置 CUDA 显存峰值统计，便于稍后测量本次生成实际消耗的峰值显存
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


# 逐 token 消费生成器，每产出一个新 token 就立即解码并打印，形成流式输出效果
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    stop_token_ids=stop_ids
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

# 生成结束后，打印本次推理过程中使用到的 GPU 显存峰值
if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")